<a href="https://colab.research.google.com/github/VainaviS/EndoNet/blob/main/notebooks/yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Pip install method (recommended)

!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output()

# prevent ultralytics from tracking your activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import YOLO

from IPython.display import display, Image

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!unzip Glenda_v1.5_classes.zip

In [ ]:
import os

os.makedirs("glenda_yolo/images/train", exist_ok=True)
os.makedirs("glenda_yolo/images/val", exist_ok=True)
os.makedirs("glenda_yolo/images/test", exist_ok=True)

os.makedirs("glenda_yolo/labels/train", exist_ok=True)
os.makedirs("glenda_yolo/labels/val", exist_ok=True)
os.makedirs("glenda_yolo/labels/test", exist_ok=True)

In [ ]:
import random
import shutil
from pathlib import Path

image_dir = Path("Glenda_v1.5_classes/annots")
images = list(image_dir.glob("*.jpg"))

random.shuffle(images)

train_split = int(0.7 * len(images))
val_split = int(0.9 * len(images))

train_images = images[:train_split]
val_images = images[train_split:val_split]
test_images = images[val_split:]

In [ ]:
def move_images(img_list, folder):
    for img in img_list:
        shutil.copy(img, f"glenda_yolo/images/{folder}/{img.name}")

move_images(train_images, "train")
move_images(val_images, "val")
move_images(test_images, "test")

In [ ]:
!pip install ultralytics pycocotools

In [ ]:
import json
import os

with open("Glenda_v1.5_classes/coco.json") as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]}
annotations = coco["annotations"]

os.makedirs("labels", exist_ok=True)

for ann in annotations:

    img_info = images[ann["image_id"]]
    width = img_info["width"]
    height = img_info["height"]
    filename = img_info["file_name"].replace(".jpg",".txt")

    label_path = os.path.join("labels", filename)

    # segmentation polygon
    seg = ann["segmentation"][0]

    xs = seg[0::2]
    ys = seg[1::2]

    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    box_w = x_max - x_min
    box_h = y_max - y_min

    x_center = (x_min + box_w/2) / width
    y_center = (y_min + box_h/2) / height
    box_w /= width
    box_h /= height

    cls = ann["category_id"] - 1

    with open(label_path, "a") as f:
        f.write(f"{cls} {x_center} {y_center} {box_w} {box_h}\n")

In [ ]:
import os
import random
import shutil

images_dir = "Glenda_v1.5_classes/frames"
labels_dir = "labels"

images = os.listdir(images_dir)
random.shuffle(images)

train_split = int(0.7 * len(images))
val_split = int(0.9 * len(images))

train = images[:train_split]
val = images[train_split:val_split]
test = images[val_split:]

def move(files, split):

    for f in files:

        shutil.copy(f"{images_dir}/{f}", f"glenda_yolo/images/{split}/{f}")

        label = f.replace(".jpg",".txt")

        if os.path.exists(f"{labels_dir}/{label}"):

            shutil.copy(f"{labels_dir}/{label}", f"glenda_yolo/labels/{split}/{label}")

move(train,"train")
move(val,"val")
move(test,"test")

In [ ]:
!head glenda_yolo/labels/train/*.txt

In [ ]:
import cv2
import random
import os
import matplotlib.pyplot as plt

image_dir = "Glenda_v1.5_classes/frames"
label_dir = "labels"

images = os.listdir(image_dir)
sample_images = random.sample(images, 5)

for img_name in sample_images:

    img_path = os.path.join(image_dir, img_name)
    label_path = os.path.join(label_dir, img_name.replace(".jpg",".txt"))

    img = cv2.imread(img_path)
    h, w, _ = img.shape

    if os.path.exists(label_path):

        with open(label_path) as f:
            lines = f.readlines()

        for line in lines:

            cls, xc, yc, bw, bh = map(float, line.split())

            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)

            cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(img,str(int(cls)),(x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

    plt.figure(figsize=(6,4))
    plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    plt.axis("off")

In [ ]:
!ls glenda_yolo/images/train | wc -l
!ls glenda_yolo/images/val | wc -l
!ls glenda_yolo/images/test | wc -l

In [ ]:
yaml_content = """
path: /content/glenda_yolo

train: images/train
val: images/val
test: images/test

names:
  0: peritoneum
  1: ovary
  2: tie
  3: uterus
"""

with open("glenda.yaml","w") as f:
    f.write(yaml_content)

In [ ]:
from ultralytics import YOLO

# Upgrade numpy and reinstall ultralytics to fix potential binary incompatibility
!pip install --upgrade numpy
!pip install --upgrade ultralytics

model = YOLO("yolov8m.pt")

model.train(
    data="/content/glenda.yaml",
    epochs=100,
    imgsz=640,
    batch=16
)

In [ ]:
model.predict(
    source="glenda_yolo/images/test",
    conf=0.25,
    save=True
)

In [ ]:
!ls glenda_yolo/images/train | wc -l
!ls glenda_yolo/labels/train | wc -l

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

test_dir = "/content/glenda_yolo/images/test"
pred_dir = "/content/runs/detect/predict"

images = os.listdir(test_dir)

for img_name in images[:5]:

    orig = cv2.imread(os.path.join(test_dir, img_name))
    pred = cv2.imread(os.path.join(pred_dir, img_name))

    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    pred = cv2.cvtColor(pred, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.title("Original")
    plt.imshow(orig)
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title("YOLO Detection")
    plt.imshow(pred)
    plt.axis("off")

    plt.show()